# Deploy Fraud Investigation Agent to AgentCore Runtime

This notebook deploys the fraud investigation agent using the
`bedrock-agentcore-starter-toolkit` with **direct code deployment**
(no Docker build required).

Run each cell in order with **Shift+Enter**.

In [ ]:
# Cell 1 — Install dependencies
# After this cell completes, restart the kernel:
#   Kernel > Restart Kernel
# Then continue from Cell 2.

import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-cache-dir',
     'strands-agents==1.20.0', 'strands-agents-tools==0.2.15',
     'boto3>=1.40.75', 'botocore>=1.40.75',
     'bedrock-agentcore==1.0.6', 'bedrock-agentcore-starter-toolkit==0.1.33'],
    capture_output=True, text=True
)
# Show only real errors, not pip dependency warnings
errors = [l for l in result.stderr.splitlines()
          if l and 'dependency' not in l.lower()
          and 'requires' not in l.lower()
          and 'incompatible' not in l.lower()
          and 'WARNING' not in l
          and 'Note:' not in l]
if errors:
    print('\n'.join(errors))
print('\n✅ Dependencies installed.')
print('⚠️  Restart the kernel now: Kernel → Restart Kernel')

---
### ⚠️ Restart the kernel, then run Cell 2 onwards

In [ ]:
%%writefile runtime.py
import os
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

# Lazy init — agent is created on first request to stay
# within the 30-second runtime init timeout.
_agent = None


def get_agent():
    global _agent
    if _agent is None:
        import json, boto3
        from strands import Agent
        from strands.tools import tool
        from strands.models.bedrock import BedrockModel

        BEDROCK_MODEL_ID = os.environ.get(
            'BEDROCK_MODEL_ID',
            'anthropic.claude-3-5-sonnet-20241022-v2:0'
        )
        KNOWLEDGE_BASE_ID = os.environ.get(
            'KNOWLEDGE_BASE_ID', ''
        )
        AWS_REGION = os.environ.get('AWS_REGION', 'us-west-2')

        bedrock_rt = boto3.client(
            'bedrock-agent-runtime', region_name=AWS_REGION
        )

        @tool
        def search_fraud_playbooks(query: str) -> str:
            """Search fraud detection playbooks and response
            procedures using the Bedrock Knowledge Base.

            Args:
                query: The search query string.

            Returns:
                Relevant playbook excerpts.
            """
            try:
                resp = bedrock_rt.retrieve(
                    knowledgeBaseId=KNOWLEDGE_BASE_ID,
                    retrievalQuery={'text': query},
                    retrievalConfiguration={
                        'vectorSearchConfiguration': {
                            'numberOfResults': 3
                        }
                    }
                )
                results = [
                    r.get('content', {}).get('text', '')
                    for r in resp.get('retrievalResults', [])
                ]
                return '\n---\n'.join(results) if results \
                    else 'No relevant playbooks found.'
            except Exception as e:
                return f'Error: {str(e)}'

        SYSTEM_PROMPT = (
            'You are an AI Fraud Investigation Agent for '
            'ShopSmart. Investigate flagged transactions by '
            'gathering transaction details, customer profiles, '
            'login activity, and support case history using MCP '
            'tools. Consult fraud playbooks via the Knowledge '
            'Base. Correlate all signals and take protective '
            'action when warranted. Always report: '
            'classification, confidence level, anomaly signals '
            'found, action taken, and recommendations.'
        )

        model = BedrockModel(
            model_id=BEDROCK_MODEL_ID,
            region_name=AWS_REGION,
            max_tokens=4096
        )

        _agent = Agent(
            model=model,
            tools=[search_fraud_playbooks],
            system_prompt=SYSTEM_PROMPT
        )
    return _agent


@app.entrypoint
def invoke(payload):
    """AgentCore Runtime entrypoint function."""
    agent = get_agent()
    user_input = payload.get('prompt', '')
    response = agent(user_input)
    return response.message['content'][0]['text']


if __name__ == '__main__':
    app.run()

In [ ]:
%%writefile runtime.py
import os, json, boto3
from strands import Agent
from strands.tools import tool
from strands.models.bedrock import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp

# Configuration from environment variables
BEDROCK_MODEL_ID = os.environ.get(
    'BEDROCK_MODEL_ID',
    'anthropic.claude-3-5-sonnet-20241022-v2:0'
)
KNOWLEDGE_BASE_ID = os.environ.get('KNOWLEDGE_BASE_ID', '')
AWS_REGION = os.environ.get('AWS_REGION', 'us-west-2')

bedrock_agent_runtime = boto3.client(
    'bedrock-agent-runtime', region_name=AWS_REGION
)


@tool
def search_fraud_playbooks(query: str) -> str:
    """Search fraud detection playbooks and response
    procedures using the Bedrock Knowledge Base.

    Args:
        query: The search query string.

    Returns:
        Relevant playbook excerpts or an error message.
    """
    try:
        response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=KNOWLEDGE_BASE_ID,
            retrievalQuery={'text': query},
            retrievalConfiguration={
                'vectorSearchConfiguration': {
                    'numberOfResults': 3
                }
            }
        )
        results = [
            r.get('content', {}).get('text', '')
            for r in response.get('retrievalResults', [])
        ]
        return '\n---\n'.join(results) if results \
            else 'No relevant playbooks found.'
    except Exception as e:
        return f'Error: {str(e)}'


SYSTEM_PROMPT = """You are an AI Fraud Investigation Agent
for ShopSmart. Investigate flagged transactions by gathering
transaction details, customer profiles, login activity, and
support case history using MCP tools. Consult fraud playbooks
via the Knowledge Base. Correlate all signals and take
protective action when warranted.

Always report: classification, confidence level, anomaly
signals found, action taken, and recommendations."""

# --- Create Agent ---

model = BedrockModel(
    model_id=BEDROCK_MODEL_ID,
    region_name=AWS_REGION,
    max_tokens=4096
)

agent = Agent(
    model=model,
    tools=[search_fraud_playbooks],
    system_prompt=SYSTEM_PROMPT
)

# --- AgentCore Runtime Setup ---

app = BedrockAgentCoreApp()


@app.entrypoint
def invoke(payload):
    """AgentCore Runtime entrypoint function."""
    user_input = payload.get('prompt', '')
    response = agent(user_input)
    return response.message['content'][0]['text']


if __name__ == '__main__':
    app.run()

In [ ]:
%%writefile runtime_requirements.txt
strands-agents==1.20.0
strands-agents-tools==0.2.15
boto3>=1.40.75
botocore>=1.40.75
bedrock-agentcore==1.0.6

In [ ]:
# Cell 5 — Install uv (required for direct code deployment)
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] += ':/root/.local/bin:/home/ec2-user/.local/bin'
print(f'\nuv version: ', end='')
!uv --version

In [ ]:
# Cell 6 — Configure the AgentCore Runtime
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint='runtime.py',
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file='runtime_requirements.txt',
    region=REGION,
    agent_name='fraud_investigation_agent',
    memory_mode='STM_ONLY',
    deployment_type='direct_code_deploy',
    runtime_type='PYTHON_3_12',
    protocol='HTTP'
)

print('Configuration completed:', response)

In [ ]:
# Cell 7 — Launch (deploys the agent, takes 3-5 minutes)
launch_result = agentcore_runtime.launch(
    env_vars={
        'KNOWLEDGE_BASE_ID': KNOWLEDGE_BASE_ID,
        'AWS_REGION': REGION,
        'BEDROCK_MODEL_ID': 'anthropic.claude-3-5-sonnet-20241022-v2:0'
    }
)

AGENT_RUNTIME_ARN = launch_result.agent_arn

print(f'Launch completed ✅')
print(f'  Agent ARN: {AGENT_RUNTIME_ARN}')

In [ ]:
# Cell 8 — Check status
status = agentcore_runtime.status()
print(f'Runtime status: {status}')

In [ ]:
# Cell 9 — Test the deployed agent
response = agentcore_runtime.invoke(
    payload={
        'prompt': (
            'Investigate flagged transaction TXN-7892 '
            'and determine if this is fraudulent'
        )
    }
)

print('🎉 Response from deployed AgentCore Runtime:\n')
print(response)

---
## Cleanup

Run the cell below to destroy the AgentCore Runtime, then
delete the CloudFormation stack from the console.

In [ ]:
# Cleanup — Destroy AgentCore Runtime
try:
    agentcore_runtime.destroy()
    print('AgentCore Runtime destroyed ✅')
except Exception as e:
    print(f'AgentCore cleanup: {e}')

# Clean up local files
import os
for f in ['runtime.py', 'runtime_requirements.txt']:
    if os.path.exists(f):
        os.remove(f)
print('Local files cleaned up ✅')
print('\n🧹 Agent cleanup complete!')
print('Now delete the CloudFormation stack from the console.')